In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
!pip install biopython
from Bio.PDB import PDBParser
from torch.utils.data import Dataset, DataLoader
import sys
sys.path.append('scripts')


'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import pdb_to_graph
import mldft_surrogate
import verify_twin_pipeline
import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper

In [3]:
#These are the target values cnn is trying to learn (correct coeffients but are dummy values)
target_coefficients = [0.12, -0.03, 0.05, 0.08, 0.01, -0.02, 0.04, 0.03, -0.01,0.06 ]
# 10 = 4 site energies and 6 coupling strengths

In [4]:
class ProteinDataset(Dataset):
    def __init__(self, tensors, labels):
        #tensor
        self.tensors = tensors
        #coefficients 
        self.labels = labels
    #number of proteins  
    def __len__(self):
        return len(self.tensors)
    #cinverting numpy array to tensor 
    def __getitem__(self, index):
        x = torch.tensor(self.tensors[index], dtype=torch.float32)
        y = torch.tensor(self.labels[index],dtype=torch.float32)
        return x, y

In [5]:
proteins = ["proteins/1ACX.pdb","proteins/1DMC.pdb","proteins/4U7S.pdb", "proteins/4UT7.pdb", "proteins/7F07.pdb",]
# stores input protein tensors and labels (x,y)
X = []
Y = []

for pdb in proteins:
    #convert to voxel grid
    tensor = pdb_voxelizier.pdb_to_tensor(pdb,grid_size=32)
    #store tensor
    X.append(tensor[0])
    # store labels, but dont have correct balues so we use dummy values 
    Y.append([target_coefficients])
X = np.array(X)
Y = np.array(Y)

In [6]:
#creating dataloader
dataset = ProteinDataset(X,Y)
#now automatcally loads proteins and shuffles order 
loader = DataLoader(dataset,batch_size=1,shuffle=True)

In [7]:
#create cnn
model = cnn_mlp_encoder.ProteinPhysicsEncoder(num_sites=4)
print(model)

ProteinPhysicsEncoder(
  (conv1): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (pool1): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (pool2): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=16384, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [8]:
#mean squared error : helps keep error positive 
criterion = nn.MSELoss()
#just using adam for now
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
#training loop
#epoch = iterations 
epochs = 100
for epoch in range(epochs):
    #track loss
    total_loss = 0
    for voxel, target in loader:
        # cnn prediction
        prediction = model(voxel)
        # Compare prediction with target coefficients 
        loss = criterion(prediction,target)
        # remove old gradients 
        optimizer.zero_grad()
        #calculate how values should change
        loss.backward()
        #update values 
        optimizer.step()
        total_loss += loss.item()
    #Note: ideally loss should decrease overtime
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.6f}")

c:\Users\Aaliyah\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\nn\modules\loss.py:630: UserWarning: Using a target size (torch.Size([1, 1, 10])) that is different to the input size (torch.Size([1, 10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1/100, Loss: 0.849125
Epoch 2/100, Loss: 0.063972
Epoch 3/100, Loss: 0.012657
Epoch 4/100, Loss: 0.004442
Epoch 5/100, Loss: 0.001557
Epoch 6/100, Loss: 0.000611
Epoch 7/100, Loss: 0.000289
Epoch 8/100, Loss: 0.000152
Epoch 9/100, Loss: 0.000087
Epoch 10/100, Loss: 0.000051
Epoch 11/100, Loss: 0.000040
Epoch 12/100, Loss: 0.000030
Epoch 13/100, Loss: 0.000018
Epoch 14/100, Loss: 0.000014
Epoch 15/100, Loss: 0.000010
Epoch 16/100, Loss: 0.000007
Epoch 17/100, Loss: 0.000005
Epoch 18/100, Loss: 0.000005
Epoch 19/100, Loss: 0.000003
Epoch 20/100, Loss: 0.000002
Epoch 21/100, Loss: 0.000002
Epoch 22/100, Loss: 0.000002
Epoch 23/100, Loss: 0.000001
Epoch 24/100, Loss: 0.000001
Epoch 25/100, Loss: 0.000001
Epoch 26/100, Loss: 0.000001
Epoch 27/100, Loss: 0.000001
Epoch 28/100, Loss: 0.000001
Epoch 29/100, Loss: 0.000000
Epoch 30/100, Loss: 0.000000
Epoch 31/100, Loss: 0.000000
Epoch 32/100, Loss: 0.000000
Epoch 33/100, Loss: 0.000000
Epoch 34/100, Loss: 0.000000
Epoch 35/100, Loss: 0.0

In [13]:
torch.save(model.state_dict(),"protein_cnn.pth")